In [1]:
import os
import cv2
import numpy as np
import scipy.io
from scipy.spatial import KDTree
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Reshape
from tensorflow.keras.applications import VGG16
from tensorflow.keras.initializers import RandomNormal
from tensorflow.keras.optimizers import Adam

# Check if GPU is detected
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [2]:

# input - image, output - image
def pad_to_multiple(image, factor=32):
    h, w = image.shape[:2]

    # Calculate how many pixels to add
    pad_h = (factor - (h % factor)) % factor
    pad_w = (factor - (w % factor)) % factor

    # Apply padding to the bottom and right only
    # (top, bottom, left, right)
    padded_img = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w,
                                    cv2.BORDER_CONSTANT, value=0)

    return padded_img


In [3]:
def get_base_model():
    base_model = VGG16(weights='imagenet', input_shape=(None, None, 3), include_top=False)

    # Start fully frozen — will unfreeze in phase 2 of training
    for layer in base_model.layers[:10]:
        layer.trainable = False

    block4_conv3 = base_model.get_layer("block4_conv3").output
    return tf.keras.Model(inputs=base_model.inputs, outputs=block4_conv3)

In [4]:
inputs = tf.keras.Input(shape=(None, None, 3))

x = get_base_model()(inputs) # Removed [0] to keep the batch dimension

init = RandomNormal(stddev=0.01)

x = Conv2D(512, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(512, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(512, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(256, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(128, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(64, (3, 3), activation='relu', dilation_rate=2, kernel_initializer=init, padding='same')(x)
x = Conv2D(1, (1, 1), activation='relu', dilation_rate=1, kernel_initializer=init, padding='same')(x)

# To remove the last dimension (channel) while keeping batch/height/width
out = x[..., 0]

model = tf.keras.Model(inputs=inputs, outputs=out)
model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ functional (Functional)         │ (None, None, None,     │     7,635,264 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, None, None,     │     1,179,904 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, None, None,     │       295,040 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, None, None, 64) │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, None, None, 1)  │            65 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, None, None)     │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,263,489 (62.04 MB)

 Trainable params: 14,528,001 (55.42 MB)

 Non-trainable params: 1,735,488 (6.62 MB)

FOR INFERENCE ON VIDEO

In [10]:
import cv2
import numpy as np
import tensorflow as tf
from IPython.display import display, clear_output
import ipywidgets as widgets
from google.colab.patches import cv2_imshow
import time
import matplotlib.cm as cm  # ADD THIS IMPORT

# ── CONFIG ────────────────────────────────────────────────────────────────
VIDEO_PATH        = '/content/Railway-platform-video.mp4'
OUTPUT_PATH       = '/content/Real-time-count.mp4'
DENSITY_OUTPUT_PATH = '/content/Real-time-density.mp4'  # ADD THIS
FRAME_SKIP        = 5
SUBSAMPLING_FACTOR = 8
# ─────────────────────────────────────────────────────────────────────────

model.load_weights('/content/CSRNet.weights.h5')
print('Weights loaded.')

def preprocess_frame(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    im_array  = frame_rgb.astype(np.float32) / 255.0
    im_array[:,:,0] = (im_array[:,:,0] - np.mean(im_array[:,:,0])) / (np.std(im_array[:,:,0]) + 1e-8)
    im_array[:,:,1] = (im_array[:,:,1] - np.mean(im_array[:,:,1])) / (np.std(im_array[:,:,1]) + 1e-8)
    im_array[:,:,2] = (im_array[:,:,2] - np.mean(im_array[:,:,2])) / (np.std(im_array[:,:,2]) + 1e-8)
    padded = pad_to_multiple(im_array, factor=32)
    return padded

def predict_count_and_map(frame_bgr):             # CHANGED: returns both count and density map
    padded   = preprocess_frame(frame_bgr)
    X_tensor = tf.constant(np.expand_dims(padded, axis=0), dtype=tf.float32)
    y_pred   = model(X_tensor, training=False)
    density_map = y_pred[0].numpy()               # (H/8, W/8)
    count    = float(tf.reduce_sum(density_map).numpy())
    return max(0, round(count)), density_map      # CHANGED: return map too

def render_count(frame_bgr, count):
    out   = frame_bgr.copy()
    label = f'Count: {count}'
    font  = cv2.FONT_HERSHEY_DUPLEX
    scale = 1.8
    thick = 3
    (tw, th), baseline = cv2.getTextSize(label, font, scale, thick)
    pad   = 18
    x1, y1 = 20, 20
    x2, y2 = x1 + tw + pad*2, y1 + th + pad*2 + baseline
    overlay = out.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, out, 0.45, 0, out)
    cv2.putText(out, label, (x1 + pad, y1 + pad + th),
                font, scale, (255, 255, 255), thick, cv2.LINE_AA)
    return out

def render_density(density_map, frame_w, frame_h, count):   # ADD THIS FUNCTION
    """Convert density map to a jet heatmap BGR frame, resized to original video dimensions."""

    # Normalise to 0-1 for colormap
    d_min, d_max = density_map.min(), density_map.max()
    if d_max > d_min:
        normalised = (density_map - d_min) / (d_max - d_min)
    else:
        normalised = np.zeros_like(density_map)

    # Apply jet colormap → RGBA float (0-1)
    heatmap_rgba = cm.jet(normalised)                        # (H, W, 4)
    heatmap_rgb  = (heatmap_rgba[:, :, :3] * 255).astype(np.uint8)  # (H, W, 3) RGB
    heatmap_bgr  = cv2.cvtColor(heatmap_rgb, cv2.COLOR_RGB2BGR)

    # Resize to match original video frame size
    heatmap_bgr  = cv2.resize(heatmap_bgr, (frame_w, frame_h),
                               interpolation=cv2.INTER_LINEAR)

    # Overlay count text on density video too
    label = f'Count: {count}'
    font  = cv2.FONT_HERSHEY_DUPLEX
    scale = 1.8
    thick = 3
    (tw, th), baseline = cv2.getTextSize(label, font, scale, thick)
    pad   = 18
    x1, y1 = 20, 20
    x2, y2 = x1 + tw + pad*2, y1 + th + pad*2 + baseline
    overlay = heatmap_bgr.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, heatmap_bgr, 0.45, 0, heatmap_bgr)
    cv2.putText(heatmap_bgr, label, (x1 + pad, y1 + pad + th),
                font, scale, (255, 255, 255), thick, cv2.LINE_AA)

    return heatmap_bgr

# ── Process video ─────────────────────────────────────────────────────────
cap          = cv2.VideoCapture(VIDEO_PATH)
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'\nVideo info:')
print(f'  Resolution : {W} x {H}')
print(f'  FPS        : {fps:.1f}')
print(f'  Frames     : {total_frames}')
print(f'  Duration   : {total_frames/fps:.1f}s')
print(f'  Frame skip : {FRAME_SKIP}  (processing {total_frames//FRAME_SKIP} frames)')
print()

fourcc         = cv2.VideoWriter_fourcc(*'mp4v')
out_writer     = cv2.VideoWriter(OUTPUT_PATH,         fourcc, fps, (W, H))
density_writer = cv2.VideoWriter(DENSITY_OUTPUT_PATH, fourcc, fps, (W, H))  # ADD THIS

frame_idx  = 0
last_count = 0
last_density_map = np.zeros((H // SUBSAMPLING_FACTOR,
                              W // SUBSAMPLING_FACTOR))   # ADD THIS: cache last density map

start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        last_count, last_density_map = predict_count_and_map(frame)  # CHANGED

    # Write count video
    rendered = render_count(frame, last_count)
    out_writer.write(rendered)

    # Write density video                                              # ADD THIS BLOCK
    density_frame = render_density(last_density_map, W, H, last_count)
    density_writer.write(density_frame)

    if frame_idx % 30 == 0:
        progress = (frame_idx / max(total_frames, 1)) * 100
        print(f'\r  Processing frame {frame_idx}/{total_frames} '
              f'({progress:.1f}%)  count={last_count}', end='')

    frame_idx += 1

cap.release()
out_writer.release()
density_writer.release()   # ADD THIS

end_time = time.time()
total_execution_time = end_time - start_time

print(f'\n\nDone!')
print(f'Count video saved    : {OUTPUT_PATH}')
print(f'Density video saved  : {DENSITY_OUTPUT_PATH}')   # ADD THIS
print(f'Total frames         : {frame_idx}')
print(f'Frames with inference: {frame_idx // FRAME_SKIP}')
print(f'Total execution time : {total_execution_time:.2f} seconds')

Weights loaded.

Video info:
  Resolution : 1080 x 1920
  FPS        : 60.0
  Frames     : 457
  Duration   : 7.6s
  Frame skip : 5  (processing 91 frames)

  Processing frame 450/457 (98.5%)  count=62

Done!
Count video saved    : /content/Real-time-count.mp4
Density video saved  : /content/Real-time-density.mp4
Total frames         : 457
Frames with inference: 91
Total execution time : 78.94 seconds
